Digital Business University of Applied Sciences

Data Science und Management (M. Sc.)

DAMI01 / DATA01 Data Analytics

Prof. Dr. Daniel Ambach

Julia Schmid (200022)

***
# Time-Series-Clustering
# Teil 2: Data Preparation
***


In [34]:
# Imports
import pandas as pd
import numpy as np
from ydata_profiling import ProfileReport
import webbrowser
import os

from parameter import *
from funktionen import * 

In [35]:
# Einstellung, sodass alle Spalten eines Datensatzes angezeigt werden
pd.set_option('display.max_columns', None) 

In [36]:
# Daten laden
path_temp = INPUT_PATH + "/data_acquisition.csv"
df = pd.read_csv(path_temp)

***
## Überblick


In [37]:
# Anzahl der Zeilen und Spalten ausgeben
print(f'Anzahl Zeilen: {df.shape[0]}')
print(f'Anzahl Spalten: {df.shape[1]}')

Anzahl Zeilen: 5051930
Anzahl Spalten: 39


In [38]:
# Zehn zufällige Einträge ausgeben
df.sample(n=10)

,id_transaction,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,description,id_cards,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web,id_user,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
4513458,8868424,2010-12-08 16:35:00,1256,236,$19.55,Online Transaction,39021,ONLINE,NaN,NaN,4784,NaN,Tolls and Bridge Fees,5160,Mastercard,Credit,5948729602096653,06/2015,95,YES,1,$15700,10/2009,2010,No,1256,37,68,1982,9,Male,1723 North Lane,41.20,-73.13,$27589,$56248,$149587,728,4
1784613,8022954,2010-05-18 13:00:00,27,2631,$169.16,Swipe Transaction,74934,Oakland,CA,94606.0,3596,NaN,Miscellaneous Machinery and Parts Manufacturing,5383,Visa,Debit,4102856768520788,05/2022,534,YES,1,$15082,11/2002,2006,No,27,78,63,1941,8,Female,3553 Mountain View Drive,32.28,-90.00,$22304,$23821,$22427,613,8
4690783,8924058,2010-12-21 22:22:00,48,5863,$73.80,Swipe Transaction,50783,Albuquerque,NM,87106.0,5411,NaN,"Grocery Stores, Supermarkets",1294,Visa,Debit,4441378583955445,12/2022,568,YES,2,$15018,02/2011,2011,No,48,69,63,1950,4,Male,5233 Valley Stream Street,35.11,-106.62,$20993,$16542,$36831,733,3
4660611,8914572,2010-12-19 17:29:00,1060,4291,$28.14,Swipe Transaction,75936,Indianapolis,IN,46221.0,5814,NaN,Fast Food Restaurants,5326,Visa,Credit,4542199497010950,05/2022,720,YES,2,$6500,10/2017,2017,No,1060,76,68,1944,2,Male,359 Valley Street,39.77,-86.14,$17909,$21547,$16040,785,5
1089771,7808994,2010-03-26 10:46:00,1195,3309,$371.00,Swipe Transaction,59474,Hamilton,OH,45011.0,3722,NaN,Passenger Railways,4459,Visa,Debit,4044587617721396,11/2021,838,YES,1,$22765,08/2016,2016,No,1195,75,68,1944,7,Male,1222 First Drive,44.01,-92.47,$23553,$40696,$15691,801,6
36808,7486548,2010-01-03 19:40:00,1098,4626,$-63.00,Swipe Transaction,50867,Hawarden,IA,51023.0,5541,NaN,Service Stations,1320,Mastercard,Debit,5740589965857604,11/2022,40,YES,1,$11121,02/2013,2013,No,1098,50,71,1969,3,Male,772 Fifth Boulevard,42.67,-95.30,$16901,$34456,$54634,752,4
3062998,8417462,2010-08-22 12:52:00,509,4588,$1.78,Swipe Transaction,20519,Bruceton Mills,WV,26525.0,5942,NaN,Book Stores,4588,Visa,Debit,4262181069766792,07/2022,519,YES,1,$12721,09/2005,2015,No,509,33,66,1986,7,Male,239 Sussex Drive,38.41,-82.43,$21842,$44534,$107410,702,4
2271032,8172856,2010-06-24 09:04:00,576,5400,$242.00,Swipe Transaction,15426,Mesa,AZ,85201.0,3390,NaN,Miscellaneous Metalwork,5664,Visa,Debit,4869825437648368,02/2022,137,YES,1,$26471,11/2011,2011,No,576,68,64,1951,6,Female,478 Hill Avenue,33.95,-84.54,$30747,$63502,$24100,638,7
4622839,8902885,2010-12-16 21:03:00,323,4268,$-71.00,Swipe Transaction,61195,Fresno,CA,93726.0,5541,NaN,Service Stations,5977,Mastercard,Debit,5365700074909909,12/2020,476,YES,1,$9513,12/2009,2009,No,323,54,66,1965,7,Male,782 Hill Lane,36.60,-119.75,$13093,$26696,$0,683,5
4252204,8786650,2010-11-19 08:46:00,662,2638,$223.15,Online Transaction,37387,ONLINE,NaN,NaN,4899,NaN,"Cable, Satellite, and Other Pay Television Ser...",2638,Mastercard,Debit,5178588521108553,04/2021,647,YES,2,$28747,04/2010,2010,No,662,32,71,1987,11,Female,670 Tenth Drive,41.86,-88.06,$42028,$85694,$120733,822,2


In [39]:
# Datensatz-Info ausgeben
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5051930 entries, 0 to 5051929
Data columns (total 39 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   id_transaction         int64  
 1   date                   object 
 2   client_id              int64  
 3   card_id                int64  
 4   amount                 object 
 5   use_chip               object 
 6   merchant_id            int64  
 7   merchant_city          object 
 8   merchant_state         object 
 9   zip                    float64
 10  mcc                    int64  
 11  errors                 object 
 12  description            object 
 13  id_cards               int64  
 14  card_brand             object 
 15  card_type              object 
 16  card_number            int64  
 17  expires                object 
 18  cvv                    int64  
 19  has_chip               object 
 20  num_cards_issued       int64  
 21  credit_limit           object 
 22  acct_open_date    

***
## Phase 3: Datenvorbereitung
***
### **Datenqualität**

### Duplikate

In [40]:
# Anzahl der Duplikate bestimmen
df_duplicates = df[df.duplicated()]
print(f'Dieser Datensatz besitz {len(df_duplicates)} Duplikate.')

Dieser Datensatz besitz 0 Duplikate.


### Fehlende Werte

In [41]:
# Fehlende Werte ermitteln
count_nan = df.isna().sum()

# Prozentsatz der NaN-Werte berechnen
percent_nan = round((count_nan / len(df)) * 100, 2)

# Werte in einem DataFrame speichern
df_nan = pd.DataFrame({
    'Anzahl NaN': count_nan,
    'Prozent NaN': percent_nan
}).sort_values(by='Anzahl NaN', ascending=False)

# Variablen mit fehlenden Werten inkl. Anzahl und Prozentanteil ausgeben
df_nan[df_nan['Anzahl NaN']>0]

,Anzahl NaN,Prozent NaN
errors,4973608,98.45
zip,574496,11.37
merchant_state,545134,10.79


Fehlende Werte treten ausschließlich bei kategorischen Variablen auf. Diese werden daher durch die Kategorien *Unknown* oder *None* ersetzt.

Bei der Variable **errors** lassen sich fehlende Werte inhaltlich als „kein Fehler aufgetreten" deuten. Sie werden daher mit dem Wert *None* gefüllt.

Bei den Variablen **zip** und **merchant_state** gibt es hingegen keinen inhaltlichen Anhaltspunkt, welcher konkrete Wert hinter den fehlenden Einträgen stecken könnte. Diese werden daher mit *Unknown* gekennzeichnet.

In [42]:
# Fehlende Werte der Variable "errors" als "None" kennzeichnen
df["errors"] = df["errors"].fillna("None")

# Variable "zip" als String speichern und fehlende Werte als "Unknown" kennzeichnen
df["zip"] = pd.to_numeric(df["zip"], errors="coerce").astype("Int64").astype("string")
df["zip"] = df["zip"].fillna("Unknown")

# Fehlende Werte der Variable "merchant_state" als "Unknown" kennzeichnen
df["merchant_state"] = df["merchant_state"].fillna("Unknown")

### Vollständigkeit

In [43]:
gesamt_zellen = df.shape[0] * df.shape[1]
fehlende_zellen = df.isna().sum().sum()
vollstaendigkeit = ((gesamt_zellen - fehlende_zellen) / gesamt_zellen) * 100
print(f"Datenvollständigkeit: {vollstaendigkeit:.2f}%")

# Quelle: Ambach, D. data analytics master. Abgerufen am 10.03.2026 von https://github.com/dan-am/data analytics master/blob/main/3 data preparation/data validation.py

Datenvollständigkeit: 100.00%


### **Irrelevante Spalten löschen**

Eine Begründung, warum diese Spalten gelöscht werden, steht im Abschnitt *Teil B: Data Understanding* in der Datei *01_ Business_Data_Understanding.ipynb*.

In [44]:
df = df.drop(
    columns=[
        "merchant_id",
        "id_cards",
        "id_user",
        "card_number",
        "expires",
        "cvv",
        "card_on_dark_web",
        "mcc",
        "card_type",
        "has_chip",
        "acct_open_date",
        "address",
        "latitude",
        "longitude",
        "current_age",
        "retirement_age",
        "merchant_city",
        "use_chip",
    ]
)

### **Kategorische Variablen**

In [45]:
# Kategorische Variablen bestimmen
categoricalVar = [col for col in df if df[col].dtype == 'object']

# Eindeutige Werte der kategorischen Variablen ausgeben
for i in categoricalVar:
    print(i)
    print(df[i].unique()) # Eindeutige Werte bestimmen
    print('')

date
['2010-01-01 00:01:00' '2010-01-01 00:02:00' '2010-01-01 00:05:00' ...
 '2011-01-17 18:10:00' '2011-01-17 18:11:00' '2011-01-17 18:12:00']

amount
['$-77.00' '$14.57' '$80.00' ... '$314.71' '$283.71' '$500.69']

merchant_state
['ND' 'IA' 'CA' 'IN' 'MD' 'NY' 'Unknown' 'TX' 'HI' 'PA' 'WI' 'GA' 'AL'
 'CT' 'WA' 'MA' 'CO' 'NJ' 'OK' 'MT' 'FL' 'AZ' 'KY' 'LA' 'IL' 'OH' 'MO'
 'MI' 'KS' 'NC' 'AR' 'TN' 'NM' 'SC' 'MN' 'NV' 'OR' 'VA' 'SD' 'WV' 'ME'
 'MS' 'RI' 'NH' 'DE' 'VT' 'Mexico' 'ID' 'NE' 'DC' 'UT' 'Vatican City' 'WY'
 'Dominican Republic' 'Canada' 'AK' 'Costa Rica' 'Germany' 'China'
 'United Kingdom' 'Estonia' 'Tuvalu' 'Taiwan' 'United Arab Emirates'
 'Lithuania' 'Netherlands' 'Japan' 'Greece' 'Vietnam' 'Haiti' 'Ireland'
 'Singapore' 'France' 'South Africa' 'Thailand' 'Italy' 'Denmark'
 'Jamaica' 'Benin' 'Belgium' 'Sierra Leone' 'Indonesia' 'Colombia'
 'Switzerland' 'Portugal' 'New Zealand' 'Jordan' 'Guatemala' 'Hong Kong'
 'Finland' 'Mongolia' 'Saudi Arabia' 'Philippines' 'Norway' 'Hunga

Die Variable **date** wird in das Datenformat Datum geändert und Tag ohne der Uhrzeitangabe (yyy-mm-dd) als seperate Variable betrachtet.

Bei den Variablen **amount**, **credit_limit**,  **per_capita_income**,  **yearly_income** und **total_debt** wird das Dollarzeichen entfernt und als numerische Variable gespeichert.

Für die Variablen **description**, **errors**,  **merchant_state** werden Kategorien gebildet, um die Werte zu gruppieren und die Auswertung zu vereinfachen.


In [46]:
# Variable "date" als Datetime speichern
df['date'] = pd.to_datetime(df['date'])

# Tag aus dem Variable "date" extrahieren
df['day'] = df['date'].dt.date

In [47]:
# Dollarzeichen entfernen und als numerische Spalten speichern
list_dollar_change = [
    "amount",
    "credit_limit",
    "per_capita_income",
    "yearly_income",
    "total_debt",
]

for i in list_dollar_change:
    df[i] = df[i].str[1:] # Dollarzeichen entfernen
    df[i] = pd.to_numeric(df[i], errors="coerce") # In numerische Spalte umwandeln

In [48]:
# Kategorien für "description" erstellen
df['description_category'] = df['description'].apply(lambda x: get_key_value(x, map_description))
df = df.drop(columns=["description"])
df["description_category"].value_counts()

description_category
Food_and_Beverage               1886273
Automotive                       930820
Retail_General                   546379
Health_and_Medical               334649
Travel_and_Transportation        324590
Financial_and_Legal              245003
Electronics_and_Technology       145586
Home_and_Garden                  127638
Entertainment_and_Recreation     105458
Utilities                         91509
Books_and_Media                   91227
Professional_Services             64883
Beauty_and_Personal_Care          54882
Metals_and_Manufacturing          50792
Clothing_and_Apparel              45687
Lodging                            6554
Name: count, dtype: int64

In [49]:
# Kategorien für "merchant_state" erstellen
df['merchant_category'] = df['merchant_state'].apply(lambda x: get_key_value(x, map_merchant_state))
df = df.drop(columns=["merchant_state"])
df['merchant_category'].value_counts()

merchant_category
US_States        4477434
North_America      14231
Europe              9174
Asia                4450
South_America        741
Oceania              390
Africa               376
Name: count, dtype: int64

In [50]:
# Kategorien für "errors" erstellen
df['error_category'] = df['errors'].apply(lambda x: get_key_value(x, map_error_category))
df = df.drop(columns=["errors"])
df["error_category"].value_counts()

error_category
No_Error           4973608
Single_Error         77971
Multiple_Errors        351
Name: count, dtype: int64

### **Numerischen Variablen**

In [51]:
# Numerische und kategorische Variablen ausgeben
numericalVar = [col for col in df if df[col].dtype != 'object']
print(numericalVar)

['id_transaction', 'date', 'client_id', 'card_id', 'amount', 'zip', 'num_cards_issued', 'credit_limit', 'year_pin_last_changed', 'birth_year', 'birth_month', 'per_capita_income', 'yearly_income', 'total_debt', 'credit_score', 'num_credit_cards']


Die Variable **zip**, **birth_year** , **birth_month**, wird als Kategorische Variable gespeichert.

In [52]:
list_numeric_change = ["zip", "birth_year", "birth_month"]
for i in list_numeric_change:
    df[i] = df[i].astype(str) # Umwandlung in String
    df[i].value_counts() # Anzahl der Werte anzeigen

### **Cleaned-Datensatz**

In [53]:
print("Neuer Cleaned-Datensatz:", df.shape)
display(df.head())
df.info()

Neuer Cleaned-Datensatz: (5051930, 22)


,id_transaction,date,client_id,card_id,amount,zip,card_brand,num_cards_issued,credit_limit,year_pin_last_changed,birth_year,birth_month,gender,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards,day,description_category,merchant_category,error_category
0,7475327,2010-01-01 00:01:00,1556,2972,-77.00,58523,Mastercard,2,24772,2010,1989,7,Female,23679,48277,110153,740,4,2010-01-01,Food_and_Beverage,US_States,No_Error
1,7475327,2010-01-01 00:01:00,1556,2972,-77.00,58523,Visa,2,15300,2020,1989,7,Female,23679,48277,110153,740,4,2010-01-01,Food_and_Beverage,US_States,No_Error
2,7475327,2010-01-01 00:01:00,1556,2972,-77.00,58523,Mastercard,2,55,2008,1989,7,Female,23679,48277,110153,740,4,2010-01-01,Food_and_Beverage,US_States,No_Error
3,7475327,2010-01-01 00:01:00,1556,2972,-77.00,58523,Amex,2,11200,2020,1989,7,Female,23679,48277,110153,740,4,2010-01-01,Food_and_Beverage,US_States,No_Error
4,7475328,2010-01-01 00:02:00,561,4575,14.57,52722,Mastercard,1,26960,2010,1971,6,Male,18076,36853,112139,834,5,2010-01-01,Retail_General,US_States,No_Error


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5051930 entries, 0 to 5051929
Data columns (total 22 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   id_transaction         int64         
 1   date                   datetime64[ns]
 2   client_id              int64         
 3   card_id                int64         
 4   amount                 float64       
 5   zip                    object        
 6   card_brand             object        
 7   num_cards_issued       int64         
 8   credit_limit           int64         
 9   year_pin_last_changed  int64         
 10  birth_year             object        
 11  birth_month            object        
 12  gender                 object        
 13  per_capita_income      int64         
 14  yearly_income          int64         
 15  total_debt             int64         
 16  credit_score           int64         
 17  num_credit_cards       int64         
 18  day                   

***
## Phase 4: Feature Engineering
***
Aus dem aufbereiteten Datensatz werden zwei Teildatensätze erstellt: eine Zeitreihe mit den Variablen card_id, day und amount, die als Grundlage für das Clustering dient, sowie ein ergänzender Datensatz mit den übrigen Variablen. Der Datensatz mit den ergänzenden Informationen wird im Anschluss herangezogen, um die gebildeten Cluster inhaltlich zu analysieren und die Ergebnisse zu bewerten.


### **Zeitreihen**

Für alle Karten-ID wird für jeden Tag ein Eintrag erstellt (vollständige Zeitreihe)


In [54]:
# Transaktionssumme pro Karte und Tag berechnen
df_timeseries_pre = (
    df.groupby(["card_id", "day"])
    .agg(amount_sum=("amount", "sum"))
    .reset_index()
)

# Datetime-Format
df_timeseries_pre["day"] = pd.to_datetime(df_timeseries_pre["day"])

# Vollständige Wertebereiche erzeugen (Min/Max Tag)
all_cards = df_timeseries_pre["card_id"].unique()
all_days = pd.date_range(
    df_timeseries_pre["day"].min(),
    df_timeseries_pre["day"].max(),
)

# Reindex durchführen
full_index = pd.MultiIndex.from_product(
    [all_cards, all_days],
    names=["card_id", "day"]
)

# An Tagen an denen es keine Transaktionen werden mit dem Wert 0 gefüllt
df_timeseries = (
    df_timeseries_pre.set_index(["card_id", "day"])
      .reindex(full_index, fill_value=0)
      .reset_index()
)

df_timeseries.head()

,card_id,day,amount_sum
0,0,2010-01-01,653.68
1,0,2010-01-02,64.80
2,0,2010-01-03,450.76
3,0,2010-01-04,0.00
4,0,2010-01-05,122.84


Für die Karten-IDs, bei denen mehr als 70 Prozent der Tageswerte den Betrag 0 aufweisen, werden aus dem Datensatz entfernt, da sie zu wenig Transaktionsaktivität zeigen und somit nur eine geringe Aussagekraft für das Clustering besitzen.

In [55]:
# Karten-IDs mit mehr als 70 % Null-Werten ermitteln und herausfiltern
df_timeseries_70 = (
    df_timeseries
    .assign(zero=df_timeseries["amount_sum"].eq(0))  # True/False für 0-Wert
    .groupby("card_id")
    .agg(
        total_days=("day", "nunique"),   # Anzahl Tage
        zero_days=("zero", "sum"),        # True zählt als 1
    )
    .assign(
        zero_percent=lambda x: 100 * x["zero_days"] / x["total_days"]
    )
    .sort_values("zero_percent", ascending=False)
)

card_ids_over_70 = (
    df_timeseries_70
    .loc[df_timeseries_70["zero_percent"] >= 30]
    .index
    .tolist()
)

df_timeseries = df_timeseries[
    df_timeseries["card_id"].isin(card_ids_over_70)
]


In [56]:
# Pivotisierung: jede Zeile repräsentiert eine Karte
# Fehlende Werte werden mit 0 gefüllt
df_timeseries = df_timeseries.pivot(
    index='card_id', columns='day', values='amount_sum'
).fillna(0)

In [57]:
# Daten speichern
path_temp = INPUT_PATH + "/data_timeseries.csv"
df_timeseries.to_csv(path_temp, index=True)

### **Zusatzinfos**
Ergänzend wird ein Zusatzinfo-Datensatz erstellt, der die übrigen Variablen pro Karte zusammenfasst. Dabei werden numerische Spalten durch ihren Median und kategoriale Spalten durch den jeweils häufigsten Wert aggregiert. Die Ergebnisse werden anschließend zu einem Datensatz zusammengeführt, sodass jede card_id durch genau eine Zeile mit zusammengefassten Informationen repräsentiert wird.

In [58]:
df_add_info = df.drop(["date", "id_transaction", "amount", "day"], axis=1)

In [59]:
# Numerische Spalten aggregieren (Median)
numericalVar = df_add_info.select_dtypes(include='number').columns
df_num = df_add_info.groupby("card_id")[numericalVar].median()

# Kategorische Spalten aggregieren (häufigster Wert)
categoricalVar = df_add_info.select_dtypes(exclude='number').columns
df_cat = df_add_info.groupby("card_id")[categoricalVar].agg(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)

# Numerische und kategorische Ergebnisse zusammenführen
df_zusatzinfo = df_num.join(df_cat)

# card_id Spalte löschen, da diese Spalte als Index gespeichert ist
df_zusatzinfo = df_zusatzinfo.drop("card_id", axis=1)

df_zusatzinfo.head()

In [61]:
# Daten speichern
path_temp = INPUT_PATH + "/data_add_info.csv"
df_zusatzinfo.to_csv(path_temp, index=True)

In [62]:
# Profilingreport der Zusatzinfos laden 
pr = ProfileReport(df_zusatzinfo, title = 'Credit Data') 
filename_pr = "../output/add_info_data_pr.html" 
path_pr = os.path.abspath(filename_pr) 

pr.to_file(path_pr)  # ProfileReport als HTML speichern
webbrowser.open(f"file://{path_pr}")  # ProfileReport im Browser öffnen

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 17/17 [00:00<00:00, 273192.21it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

True

***

In [65]:
# # Alle Variablen und Objekte aus dem Arbeitsspeicher löschen
%reset -f

***
***